# Hướng Dẫn Giải Thích Chi Tiết: `src/ai_models.py`

Notebook này phân tích cấu trúc, công dụng và cách hoạt động của mô hình tối ưu **XGBoost** và mô hình mạng nơ-ron lai ghép **Conv1D-Transformer** được xây dựng trong `src/ai_models.py`.

---

## 🔍 1. Tổng Quan Về Các Mô Hình Sử Dụng

Trong dự án này, chúng ta kết hợp hai trường phái mạnh mẽ nhất hiện nay trong Machine Learning:
1. **XGBoost (Cây Quyết Định Tối Ưu):** Cực kỳ mạnh mẽ đối với dữ liệu dạng bảng, tìm kiếm các mối quan hệ phi tuyến tính dựa trên các chỉ báo kỹ thuật và vĩ mô một cách trực tiếp.
2. **Hybrid Conv1D-Transformer (Deep Learning lai ghép):** Kết hợp lớp **Conv1D** để trích xuất đặc trưng chu kỳ ngắn cục bộ và các lớp **Attention** để nắm bắt xu hướng chuỗi thời gian dài hạn trong cửa sổ trượt 45 ngày.

In [ ]:
import sys
import os
# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath('..'))

from src.ai_models import build_xgboost_optimized, build_transformer
print("Import các hàm khởi tạo mô hình thành công!")

## ⚙️ 2. Mô Hướng Huấn Luyện XGBoost Tối Ưu Hóa Tìm Kiếm Siêu Tham Số

### Hàm `build_xgboost_optimized(X_train, y_train)`

XGBoost không nhận đầu vào dạng 3D chuỗi thời gian. Vì vậy, trước khi đưa vào huấn luyện, chúng ta làm phẳng dữ liệu (`reshape` từ 3D sang 2D):
$$\text{Shape: } [N, 45, 23] \rightarrow [N, 45 \times 24] = [N, 1080]$$
Tức là 1080 cột đặc trưng biểu thị tất cả giá trị chỉ báo của 45 ngày liên tiếp.

Hàm thực hiện tìm kiếm ngẫu nhiên tối ưu (`RandomizedSearchCV`) kết hợp phân chia chuỗi thời gian (`TimeSeriesSplit`) để tìm bộ tham số tốt nhất một cách nhanh chóng:
- `n_estimators`: Số lượng cây quyết định (từ 100 đến 300).
- `max_depth`: Độ sâu tối đa của cây (từ 3 đến 6).
- `learning_rate`: Tốc độ học (0.01 đến 0.1).
- `subsample` và `colsample_bytree`: Tỷ lệ lấy mẫu dòng/cột dùng chống quá khớp.
- `n_jobs=-1`: Chạy song song tìm kiếm trên tất cả các nhân CPU.

In [ ]:
import numpy as np
# Tạo dữ liệu ngẫu nhiên giả lập tập huấn luyện để chạy thử nghiệm XGBoost
np_random = np.random.RandomState(42)
X_train_dummy = np_random.normal(size=(50, 45, 24))
y_train_dummy = np_random.normal(size=(50,))

# Làm phẳng đầu vào cho XGBoost
X_train_dummy_flat = X_train_dummy.reshape(X_train_dummy.shape[0], -1)
print(f"Hình dạng tập train dummy làm phẳng: {X_train_dummy_flat.shape}")

# Thử nghiệm chạy RandomizedSearchCV trên dữ liệu dummy
xgb_dummy_model = build_xgboost_optimized(X_train_dummy_flat, y_train_dummy)

## 🤖 3. Kiến Trúc Mạng Nơ-ron Conv1D-Transformer Lai Ghép

### Hàm `build_transformer(input_shape)`

Kiến trúc lai ghép của chúng ta nhận đầu vào 3D trực tiếp mà không cần làm phẳng dữ liệu, cấu thành từ các thành phần sau:

1. **Lớp Đầu Vào (Input Layer):** Nhận ma trận `(time_steps=45, features=24)`.
2. **Lớp Conv1D (1D Convolution):** 128 bộ lọc với kích thước hạt kernel_size = 3 nhằm học và trích xuất các đặc trưng chu kỳ cục bộ ngắn hạn.
3. **Chuẩn Hóa Lớp (Layer Normalization):** Duy trì sự ổn định phân phối đầu ra của lớp Conv1D.
4. **Nhúng Vị Trí Thời Gian (Positional Embedding):** Thêm thông tin về thứ tự tuần tự của ngày giao dịch vào ma trận đặc trưng.
5. **Hai lớp Multi-Head Attention (8 heads, key_dim=128):** 
   - Giúp mạng nơ-ron học mối liên kết dài hạn giữa các ngày khác nhau trong quá khứ 45 phiên.
6. **Residual Connections (Kết nối tắt) & Layer Normalization:** Tránh suy giảm gradient khi lan truyền ngược qua các lớp attention sâu.
7. **Global Average Pooling & Dense Layers (Fully Connected):** Gom các chiều thời gian lại, đưa qua lớp ẩn Dense (Dropout 20% để chống quá khớp) và đưa ra giá trị thực đơn lẻ (Target Return).

In [ ]:
# Khởi tạo cấu trúc Conv1D-Transformer lai ghép
transformer_shape = (45, 24)
transformer_model = build_transformer(transformer_shape)

# Hiển thị sơ đồ kiến trúc chi tiết
transformer_model.summary()